# 07 — Model Optimization and Deployment Readiness

## Project: PlantLeafNet

### Objectives
- Measure model checkpoint sizes.
- Benchmark inference latency and throughput.
- Compare CPU and GPU inference performance.
- Investigate dynamic quantization.
- Evaluate optimization trade-offs.
- Identify a suitable model for deployment.

### Models
- MobileNetV3-Large Fine-Tuned
- EfficientNetV2-S Fine-Tuned

### Optimization techniques
- Inference benchmarking
- Dynamic quantization
- Model size comparison

The goal is to improve deployment efficiency while preserving classification performance.

In [1]:
import os
import json
import time
import copy
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from torchvision import models

from google.colab import drive
drive.mount("/content/drive")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_DIR = Path("/content/drive/MyDrive/PlantLeafNet")
MODEL_DIR = PROJECT_DIR / "models"
RESULT_DIR = PROJECT_DIR / "results"

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
Device: cuda
GPU: Tesla T4


In [2]:
with open(PROJECT_DIR / "class_mapping.json", "r") as f:
    class_to_idx = json.load(f)

num_classes = len(class_to_idx)

# MobileNetV3-Large
mobilenet = models.mobilenet_v3_large(weights=None)
mobilenet.classifier[3] = nn.Linear(
    mobilenet.classifier[3].in_features,
    num_classes
)

mobilenet.load_state_dict(
    torch.load(
        MODEL_DIR / "best_mobilenet_finetuned.pth",
        map_location="cpu",
        weights_only=True
    )
)

# EfficientNetV2-S
efficientnet = models.efficientnet_v2_s(weights=None)
efficientnet.classifier[1] = nn.Linear(
    efficientnet.classifier[1].in_features,
    num_classes
)

efficientnet.load_state_dict(
    torch.load(
        MODEL_DIR / "best_efficientnet_finetuned.pth",
        map_location="cpu",
        weights_only=True
    )
)

mobilenet.eval()
efficientnet.eval()

print("Both trained models loaded successfully!")
print("Number of classes:", num_classes)

Both trained models loaded successfully!
Number of classes: 38


In [5]:
def benchmark_model(model, device, batch_size=32, iterations=100):

    model = model.to(device)
    model.eval()

    dummy_input = torch.randn(
        batch_size, 3, 224, 224
    ).to(device)

    # Warm-up
    with torch.inference_mode():
        for _ in range(10):
            model(dummy_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        for _ in range(iterations):
            model(dummy_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    total_images = batch_size * iterations

    latency = elapsed / iterations * 1000
    throughput = total_images / elapsed

    return latency, throughput

In [6]:
gpu_results = []

for name, model in [
    ("MobileNetV3-Large", mobilenet),
    ("EfficientNetV2-S", efficientnet)
]:
    latency, throughput = benchmark_model(
        model, torch.device("cuda"),
        batch_size=32,
        iterations=100
    )

    gpu_results.append({
        "Model": name,
        "Device": "GPU (T4)",
        "Batch Size": 32,
        "Latency (ms/batch)": round(latency, 2),
        "Throughput (images/sec)": round(throughput, 2)
    })

gpu_df = pd.DataFrame(gpu_results)
gpu_df

,Model,Device,Batch Size,Latency (ms/batch),Throughput (images/sec)
0,MobileNetV3-Large,GPU (T4),32,21.54,1485.61
1,EfficientNetV2-S,GPU (T4),32,101.47,315.36


In [7]:
cpu_results = []

for name, model in [
    ("MobileNetV3-Large", mobilenet),
    ("EfficientNetV2-S", efficientnet)
]:
    latency, throughput = benchmark_model(
        model, torch.device("cpu"),
        batch_size=8,
        iterations=20
    )

    cpu_results.append({
        "Model": name,
        "Device": "CPU",
        "Batch Size": 8,
        "Latency (ms/batch)": round(latency, 2),
        "Throughput (images/sec)": round(throughput, 2)
    })

cpu_df = pd.DataFrame(cpu_results)
cpu_df

,Model,Device,Batch Size,Latency (ms/batch),Throughput (images/sec)
0,MobileNetV3-Large,CPU,8,154.73,51.70
1,EfficientNetV2-S,CPU,8,936.07,8.55


In [8]:
benchmark_df = pd.concat(
    [gpu_df, cpu_df],
    ignore_index=True
)

benchmark_df.to_csv(
    RESULT_DIR / "inference_benchmark.csv",
    index=False
)

print("Inference benchmark saved!")

Inference benchmark saved!


In [9]:
import copy

mobilenet_cpu = copy.deepcopy(mobilenet).cpu().eval()

quantized_mobilenet = torch.ao.quantization.quantize_dynamic(
    mobilenet_cpu,
    {nn.Linear},
    dtype=torch.qint8
)

print("Dynamic quantization completed!")

/tmp/ipykernel_314/330609898.py:5: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_mobilenet = torch.ao.quantization.quantize_dynamic(


Dynamic quantization completed!


In [10]:
torch.save(
    mobilenet_cpu.state_dict(),
    "/content/mobilenet_fp32.pth"
)

torch.save(
    quantized_mobilenet.state_dict(),
    "/content/mobilenet_dynamic_int8.pth"
)

fp32_size = os.path.getsize("/content/mobilenet_fp32.pth") / (1024**2)
int8_size = os.path.getsize("/content/mobilenet_dynamic_int8.pth") / (1024**2)

print(f"FP32 size: {fp32_size:.2f} MB")
print(f"Dynamic INT8 size: {int8_size:.2f} MB")
print(f"Size reduction: {(1-int8_size/fp32_size)*100:.2f}%")

FP32 size: 16.41 MB
Dynamic INT8 size: 12.76 MB
Size reduction: 22.25%


In [11]:
# Compare FP32 and quantized predictions
test_input = torch.randn(1, 3, 224, 224)

with torch.inference_mode():
    fp32_output = mobilenet_cpu(test_input)
    int8_output = quantized_mobilenet(test_input)

fp32_prediction = fp32_output.argmax(dim=1).item()
int8_prediction = int8_output.argmax(dim=1).item()

print("FP32 prediction:", fp32_prediction)
print("INT8 prediction:", int8_prediction)
print("Predictions match:", fp32_prediction == int8_prediction)

FP32 prediction: 10
INT8 prediction: 10
Predictions match: True


In [12]:
optimization_summary = pd.DataFrame([
    {
        "Model": "MobileNetV3-Large FP32",
        "Checkpoint Size (MB)": fp32_size
    },
    {
        "Model": "MobileNetV3-Large Dynamic INT8",
        "Checkpoint Size (MB)": int8_size
    }
])

optimization_summary.to_csv(
    RESULT_DIR / "quantization_results.csv",
    index=False
)

print("Quantization results saved!")

Quantization results saved!
